# PyTorch — Chapter 9: Evaluating PyTorch Models


## 1. Vì sao cần đánh giá thực nghiệm

Thiết kế model có rất nhiều điểm phải quyết định (số layer, số unit, activation, loss, optimizer, số epoch...) không có công thức đúng tuyệt đối.

## 2. Tách dữ liệu (Data Splitting)

- Cách thủ công: cắt mảng theo tỉ lệ cố định (VD 66% đầu làm train). **Rủi ro**: nếu dữ liệu gốc có thứ tự (VD đã sắp theo nhãn), tập test có thể bị lệch hoàn toàn về một lớp.
- Cách chuẩn: dùng `sklearn.model_selection.train_test_split(...)` — tự động xáo trộn (shuffle) trước khi cắt.
- Tỉ lệ phổ biến 70/30 hoặc 66/33; nếu dataset rất lớn, tỉ lệ train có thể thấp hơn (VD 30/70) miễn phần train vẫn đủ lớn.

## 3. Huấn luyện có kiểm định (Validation)

- Nếu tính accuracy bằng `X_batch`/`y_batch` (dữ liệu **đang dùng để huấn luyện**), đó là "cheating" — model có thể ghi nhớ đáp án thay vì học cách suy luận, nên accuracy đo kiểu này **không phản ánh khả năng tổng quát hoá**.
- Cách đúng: chỉ tính accuracy dùng để **theo dõi/so sánh model** từ `X_test`/`y_test`.
- Dấu hiệu overfitting: train accuracy tiếp tục tăng trong khi test accuracy giảm hoặc đứng yên.
- **Kết quả thật từ sách** (Output 9.1): test accuracy tăng dần từ `57.9%` (epoch 0) lên đỉnh `68.9%` quanh epoch 19–24, sau đó dao động giảm nhẹ về `67.3%` ở epoch 49.

## 4. K-fold Cross-Validation

- Vấn đề của 1 lần train-test split: có thể "may" hoặc "xui" với đúng cách chia đó, không chắc kết quả tổng quát.
- **K-fold**: chia dữ liệu thành k phần bằng nhau, lặp lại k lần huấn luyện — mỗi lần lấy 1 phần làm test, phần còn lại làm train. Kết quả cuối là **trung bình** và **độ lệch chuẩn** của k điểm accuracy, không phải một con số duy nhất.
- `StratifiedKFold` (thay vì `KFold` thường): đảm bảo mỗi fold giữ đúng tỉ lệ lớp như dữ liệu gốc.
- Để lặp lại việc train từ đầu k lần, code cần được **đóng gói thành hàm** `model_train(X_train, y_train, X_test, y_test)` — mỗi lần gọi tạo model mới hoàn toàn (`nn.Sequential(...)` bên trong hàm), không dùng lại model cũ.
- **Kết quả thật từ sách** (Output 9.2): 5 fold cho accuracy `[0.64, 0.67, 0.68, 0.63, 0.59]` → trung bình **64.05% ± 3.30%**. Độ lệch chuẩn nhỏ (3%) cho thấy kết quả khá ổn định giữa các fold, không phải một fold ăn may.


## 5. Vận dụng


**9.1** — Nạp dataset từ CSV

In [4]:
import numpy as np
data = np.loadtxt("pima-indians-diabetes.csv", delimiter=",")


**9.2** — Tách train/test bằng slicing thủ công (66/34)

In [5]:
import numpy as np
data = np.loadtxt("pima-indians-diabetes.csv", delimiter=",")
# find the boundary at 66% of total samples
count = len(data)
n_train = int(count * 0.66)
# split the data at the boundary
train_data = data[:n_train]
test_data = data[n_train:]


**9.3** — Tách train/test bằng train_test_split() của scikit-learn

In [6]:
import numpy as np
from sklearn.model_selection import train_test_split

data = np.loadtxt("pima-indians-diabetes.csv", delimiter=",")
train_data, test_data = train_test_split(data, test_size=0.33)


**9.4** — Tách tensor PyTorch bằng train_test_split()

In [7]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split

data = np.loadtxt("pima-indians-diabetes.csv", delimiter=",")
X = data[:, 0:8]
y = data[:, 8]
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)


**9.5–9.6** — Training loop cơ bản + báo cáo accuracy trên batch train

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm
from sklearn.model_selection import train_test_split

data = np.loadtxt("pima-indians-diabetes.csv", delimiter=",")
X = data[:, 0:8]
y = data[:, 8]
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

# loss function and optimizer
loss_fn = nn.BCELoss()  # binary cross entropy
optimizer = optim.Adam(model.parameters(), lr=0.0001)

n_epochs = 50    # number of epochs to run
batch_size = 10  # size of each batch
batches_per_epoch = len(X_train) // batch_size

for epoch in range(n_epochs):
    with tqdm.trange(batches_per_epoch, unit="batch", mininterval=0) as bar:
        bar.set_description(f"Epoch {epoch}")
        for i in bar:
            # take a batch
            start = i * batch_size
            X_batch = X_train[start:start+batch_size]
            y_batch = y_train[start:start+batch_size]
            # forward pass
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            # backward pass
            optimizer.zero_grad()
            loss.backward()
            # update weights
            optimizer.step()
            # print progress
            bar.set_postfix(
                loss=float(loss)
            )


Note: you may need to restart the kernel to use updated packages.


Epoch 49: 100%|██████████| 51/51 [00:00<00:00, 347.31batch/s, loss=0.704]


**9.8** — Code hoàn chỉnh: train + đánh giá đúng trên X_test mỗi epoch

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm
from sklearn.model_selection import train_test_split

data = np.loadtxt("pima-indians-diabetes.csv", delimiter=",")
X = data[:, 0:8]
y = data[:, 8]
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

model = nn.Sequential(
    nn.Linear(8, 12),
    nn.ReLU(),
    nn.Linear(12, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

# loss function and optimizer
loss_fn = nn.BCELoss()  # binary cross entropy
optimizer = optim.Adam(model.parameters(), lr=0.0001)

n_epochs = 50    # number of epochs to run
batch_size = 10  # size of each batch
batches_per_epoch = len(X_train) // batch_size

for epoch in range(n_epochs):
    with tqdm.trange(batches_per_epoch, unit="batch", mininterval=0) as bar:
        bar.set_description(f"Epoch {epoch}")
        for i in bar:
            # take a batch
            start = i * batch_size
            X_batch = X_train[start:start+batch_size]
            y_batch = y_train[start:start+batch_size]
            # forward pass
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            # backward pass
            optimizer.zero_grad()
            loss.backward()
            # update weights
            optimizer.step()
            # print progress
            acc = (y_pred.round() == y_batch).float().mean()
            bar.set_postfix(
                loss=float(loss),
                acc=float(acc)
            )
    # evaluate model at end of epoch
    y_pred = model(X_test)
    acc = (y_pred.round() == y_test).float().mean()
    acc = float(acc)
    print(f"End of {epoch}, accuracy {acc}")


Epoch 0: 100%|██████████| 51/51 [00:00<00:00, 351.24batch/s, acc=0.5, loss=0.747]


End of 0, accuracy 0.4055117964744568


Epoch 1: 100%|██████████| 51/51 [00:00<00:00, 318.58batch/s, acc=0.5, loss=0.818]


End of 1, accuracy 0.5472440719604492


Epoch 2: 100%|██████████| 51/51 [00:00<00:00, 352.57batch/s, acc=0.4, loss=0.976]


...

End of 47, accuracy 0.6535432934761047


Epoch 48: 100%|██████████| 51/51 [00:00<00:00, 349.72batch/s, acc=0.4, loss=0.933]


End of 48, accuracy 0.6574802994728088


Epoch 49: 100%|██████████| 51/51 [00:00<00:00, 355.31batch/s, acc=0.4, loss=0.932]

End of 49, accuracy 0.6574802994728088


**9.9–9.11** — Đóng gói training thành hàm + đánh giá bằng 5-fold cross-validation

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm
from sklearn.model_selection import StratifiedKFold

data = np.loadtxt("pima-indians-diabetes.csv", delimiter=",")
X = data[:, 0:8]
y = data[:, 8]
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

def model_train(X_train, y_train, X_test, y_test):
    # create new model
    model = nn.Sequential(
        nn.Linear(8, 12),
        nn.ReLU(),
        nn.Linear(12, 8),
        nn.ReLU(),
        nn.Linear(8, 1),
        nn.Sigmoid()
    )

    # loss function and optimizer
    loss_fn = nn.BCELoss()  # binary cross entropy
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    n_epochs = 25    # number of epochs to run
    batch_size = 10  # size of each batch
    batches_per_epoch = len(X_train) // batch_size

    for epoch in range(n_epochs):
        with tqdm.trange(batches_per_epoch, unit="batch", mininterval=0, disable=True
                        ) as bar:
            bar.set_description(f"Epoch {epoch}")
            for i in bar:
                # take a batch
                start = i * batch_size
                X_batch = X_train[start:start+batch_size]
                y_batch = y_train[start:start+batch_size]
                # forward pass
                y_pred = model(X_batch)
                loss = loss_fn(y_pred, y_batch)
                # backward pass
                optimizer.zero_grad()
                loss.backward()
                # update weights
                optimizer.step()
                # print progress
                acc = (y_pred.round() == y_batch).float().mean()
                bar.set_postfix(
                    loss=float(loss),
                    acc=float(acc)
                )
    # evaluate accuracy at end of training
    y_pred = model(X_test)
    acc = (y_pred.round() == y_test).float().mean()
    return float(acc)

# define 5-fold cross validation test harness
kfold = StratifiedKFold(n_splits=5, shuffle=True)
cv_scores = []
for train, test in kfold.split(X, y):
    # create model, train, and get accuracy
    acc = model_train(X[train], y[train], X[test], y[test])
    print("Accuracy: %.2f" % acc)
    cv_scores.append(acc)
# evaluate the model
print("%.2f%% (+/- %.2f%%)" % (np.mean(cv_scores)*100, np.std(cv_scores)*100))


Accuracy: 0.56
Accuracy: 0.62
Accuracy: 0.60
Accuracy: 0.69
Accuracy: 0.69
63.30% (+/- 4.93%)
